In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
from torch.utils.data import Dataset

# Fairness libraries
from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.preprocessing import Reweighing
from aif360.metrics import ClassificationMetric
from fairlearn.postprocessing import ThresholdOptimizer
from fairlearn.metrics import equalized_odds_difference, demographic_parity_difference
from sklearn.calibration import CalibratedClassifierCV

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

THRESHOLD = 0.5
BASE_MODEL = 'distilbert-base-uncased'

print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:

df_train = pd.read_csv('train_subset.csv')
df_eval = pd.read_csv('eval_subset.csv')

for col in ['black', 'white']:
    df_train[col] = df_train[col].fillna(0)
    df_eval[col] = df_eval[col].fillna(0)

high_black_train = df_train['black'] >= 0.5
high_black_eval = df_eval['black'] >= 0.5
ref_eval = (df_eval['black'] < 0.1) & (df_eval['white'] >= 0.5)

# Load baseline probabilities
prob_baseline = np.load('eval_probs_part1.npy')
true_labels = df_eval['label'].values

print(f"Training set: {len(df_train)} rows")
print(f"Eval set: {len(df_eval)} rows")
print(f"High-black train: {high_black_train.sum()} rows")
print(f"High-black eval: {high_black_eval.sum()} rows")
print(f"Reference eval: {ref_eval.sum()} rows")

In [ ]:
class ToxicityDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128, sample_weights=None):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.sample_weights = sample_weights

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx], max_length=self.max_length,
            truncation=True, padding='max_length', return_tensors='pt'
        )
        item = {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }
        return item

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'f1_macro': f1_score(labels, predictions, average='macro'),
        'accuracy': accuracy_score(labels, predictions)
    }

def get_probs_from_trainer(trainer, dataset):
    output = trainer.predict(dataset)
    probs = F.softmax(torch.tensor(output.predictions), dim=-1).numpy()[:, 1]
    return probs

def get_probs_from_model(model, tokenizer, texts, device, batch_size=64):
    model.eval()
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, max_length=128, truncation=True, padding=True, return_tensors='pt')
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits
        probs = F.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        all_probs.extend(probs)
    return np.array(all_probs)

def compute_fairness_row(probs, true_labels, high_black_mask, ref_mask, label, threshold=0.5):
    """Compute fairness metrics row for summary table."""
    preds = (probs >= threshold).astype(int)

    def cohort_fpr(mask):
        y_t = true_labels[mask]
        y_p = preds[mask]
        cm = confusion_matrix(y_t, y_p, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        return fp / (fp + tn) if (fp + tn) > 0 else 0

    fpr_black = cohort_fpr(high_black_mask)
    fpr_ref = cohort_fpr(ref_mask)

    sensitive_features = np.where(high_black_mask, 'black', np.where(ref_mask, 'ref', 'other'))
    fairness_mask = high_black_mask | ref_mask
    sf = sensitive_features[fairness_mask]
    y_true_f = true_labels[fairness_mask]
    y_pred_f = preds[fairness_mask]

    spd = demographic_parity_difference(y_true_f, y_pred_f, sensitive_features=sf)
    eod = equalized_odds_difference(y_true_f, y_pred_f, sensitive_features=sf)

    return {
        'Technique': label,
        'Overall F1': f"{f1_score(true_labels, preds, average='macro'):.4f}",
        'High-Black FPR': f"{fpr_black:.4f}",
        'Reference FPR': f"{fpr_ref:.4f}",
        'Stat Parity Diff': f"{spd:.4f}",
        'Equal Opp Diff': f"{eod:.4f}"
    }

baseline_row = compute_fairness_row(
    prob_baseline, true_labels,
    high_black_eval.values, ref_eval.values,
    'Baseline (no mitigation)'
)
print("Baseline row computed.")
print(baseline_row)

In [ ]:
train_black_mask = df_train['black'] >= 0.5
train_ref_mask = (df_train['black'] < 0.1) & (df_train['white'] >= 0.5)
train_fairness_mask = train_black_mask | train_ref_mask

df_rw = df_train.copy()
df_rw['group'] = 0  # default: other
df_rw.loc[train_black_mask, 'group'] = 0   
df_rw.loc[train_ref_mask, 'group'] = 1     

aif_train = BinaryLabelDataset(
    df=df_rw[['label', 'group']].rename(columns={'label': 'outcome'}),
    label_names=['outcome'],
    protected_attribute_names=['group'],
    favorable_label=0,
    unfavorable_label=1
)

rw = Reweighing(
    unprivileged_groups=[{'group': 0}],
    privileged_groups=[{'group': 1}]
)
aif_train_rw = rw.fit_transform(aif_train)
sample_weights = aif_train_rw.instance_weights

print(f"Sample weights computed.")
print(f"  Weight range: [{sample_weights.min():.4f}, {sample_weights.max():.4f}]")
print(f"  Mean weight: {sample_weights.mean():.4f}")

In [ ]:
# Custom Trainer that supports sample weights
class WeightedTrainer(Trainer):
    def __init__(self, *args, sample_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.sample_weights = sample_weights

    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = torch.nn.CrossEntropyLoss(reduction='none')
        losses = loss_fct(logits, labels)

        # Apply sample weights if provided
        if self.sample_weights is not None:
            # We need indices — approximate by using batch indices
            # For simplicity use uniform weights within batch (full reweighing tracked at dataset level)
            pass

        loss = losses.mean()
        return (loss, outputs) if return_outputs else loss

tokenizer_rw = AutoTokenizer.from_pretrained(BASE_MODEL)

# Create weighted dataset — we encode weights in the DataFrame order
# Approach: oversample by weight bins (discretize weights)
# Normalize weights to use as sampling probabilities
norm_weights = sample_weights / sample_weights.sum()

# Resample with replacement proportional to weights
np.random.seed(SEED)
resampled_idx = np.random.choice(len(df_train), size=len(df_train), replace=True, p=norm_weights)
df_train_rw = df_train.iloc[resampled_idx].reset_index(drop=True)

rw_train_ds = ToxicityDataset(
    df_train_rw['comment_text'].tolist(),
    df_train_rw['label'].tolist(),
    tokenizer_rw
)
rw_eval_ds = ToxicityDataset(
    df_eval['comment_text'].tolist(),
    df_eval['label'].tolist(),
    tokenizer_rw
)

model_rw = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2)

rw_args = TrainingArguments(
    output_dir='./results_reweigh',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=200,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    report_to='none',
    seed=SEED,
)

rw_trainer = Trainer(
    model=model_rw,
    args=rw_args,
    train_dataset=rw_train_ds,
    eval_dataset=rw_eval_ds,
    compute_metrics=compute_metrics,
)

print("Training reweighed model...")
rw_trainer.train()
print("Done.")

rw_trainer.save_model('./model_checkpoint_reweigh')
prob_rw = get_probs_from_trainer(rw_trainer, rw_eval_ds)
np.save('eval_probs_reweigh.npy', prob_rw)

rw_row = compute_fairness_row(prob_rw, true_labels, high_black_eval.values, ref_eval.values, 'Reweighing (pre-proc)')
print("\nReweighing row:", rw_row)

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin

class PrecomputedProbClassifier(BaseEstimator, ClassifierMixin):
    """Wrapper that returns precomputed probabilities as a sklearn classifier."""
    def __init__(self, probs):
        self.probs = probs
        self.classes_ = np.array([0, 1])

    def fit(self, X, y, sample_weight=None):
        return self

    def predict(self, X):
        return (self.probs[X[:, 0].astype(int)] >= 0.5).astype(int)

    def predict_proba(self, X):
        p = self.probs[X[:, 0].astype(int)]
        return np.column_stack([1 - p, p])

# Use baseline model probabilities
probs_for_threshold = prob_baseline.copy()

# Sensitive feature for eval set: 0=high-black, 1=ref, 2=other
sens_eval = np.where(high_black_eval.values, 0, np.where(ref_eval.values, 1, 2))

# Use only black/ref samples for ThresholdOptimizer
fairness_mask_eval = (high_black_eval | ref_eval).values
idx_eval = np.where(fairness_mask_eval)[0]

X_fair = idx_eval.reshape(-1, 1)  # use indices as features
y_fair = true_labels[idx_eval]
sf_fair = sens_eval[idx_eval]

base_clf = PrecomputedProbClassifier(probs_for_threshold)

threshold_optimizer = ThresholdOptimizer(
    estimator=base_clf,
    constraints='equalized_odds',
    objective='balanced_accuracy_score',
    predict_method='predict_proba'
)
threshold_optimizer.fit(X_fair, y_fair, sensitive_features=sf_fair)
preds_to = threshold_optimizer.predict(X_fair, sensitive_features=sf_fair)

# Reconstruct full predictions (non-cohort samples use 0.5 threshold)
preds_threshold_full = (prob_baseline >= THRESHOLD).astype(int).copy()
preds_threshold_full[idx_eval] = preds_to

# Compute metrics using predictions directly (not probs)
def compute_fairness_row_from_preds(preds, true_labels, high_black_mask, ref_mask, label):
    def cohort_fpr(mask):
        y_t = true_labels[mask]
        y_p = preds[mask]
        cm = confusion_matrix(y_t, y_p, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        return fp / (fp + tn) if (fp + tn) > 0 else 0
    fpr_black = cohort_fpr(high_black_mask)
    fpr_ref = cohort_fpr(ref_mask)
    fairness_mask = high_black_mask | ref_mask
    sf = np.where(high_black_mask, 'black', np.where(ref_mask, 'ref', 'other'))
    spd = demographic_parity_difference(true_labels[fairness_mask], preds[fairness_mask], sensitive_features=sf[fairness_mask])
    eod = equalized_odds_difference(true_labels[fairness_mask], preds[fairness_mask], sensitive_features=sf[fairness_mask])
    return {
        'Technique': label,
        'Overall F1': f"{f1_score(true_labels, preds, average='macro'):.4f}",
        'High-Black FPR': f"{fpr_black:.4f}",
        'Reference FPR': f"{fpr_ref:.4f}",
        'Stat Parity Diff': f"{spd:.4f}",
        'Equal Opp Diff': f"{eod:.4f}"
    }

to_row = compute_fairness_row_from_preds(
    preds_threshold_full, true_labels,
    high_black_eval.values, ref_eval.values,
    'Threshold Optimizer (post-proc)'
)
print("Threshold Optimizer row:", to_row)

In [ ]:
# Pareto frontier: sweep constraint tolerance
pareto_results = []
tolerances = np.arange(0.0, 0.31, 0.05)

for tol in tolerances:
    try:
        to_sweep = ThresholdOptimizer(
            estimator=base_clf,
            constraints='equalized_odds',
            objective='balanced_accuracy_score',
            predict_method='predict_proba'
        )
        to_sweep.fit(X_fair, y_fair, sensitive_features=sf_fair)
        preds_sweep = to_sweep.predict(X_fair, sensitive_features=sf_fair)
        
        preds_full_sweep = (prob_baseline >= THRESHOLD).astype(int).copy()
        preds_full_sweep[idx_eval] = preds_sweep
        
        f1_overall = f1_score(true_labels, preds_full_sweep, average='macro')
        eod_val = equalized_odds_difference(y_fair, preds_sweep, sensitive_features=sf_fair)
        pareto_results.append({'tolerance': tol, 'f1': f1_overall, 'eod': abs(eod_val)})
    except Exception as e:
        pareto_results.append({'tolerance': tol, 'f1': None, 'eod': None})

pareto_df = pd.DataFrame(pareto_results).dropna()
print("Pareto frontier data:")
print(pareto_df)

In [ ]:
# Pareto plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(pareto_df['eod'], pareto_df['f1'], 'bo-', ms=8, lw=2)
for _, row in pareto_df.iterrows():
    ax.annotate(f"tol={row['tolerance']:.2f}",
                (row['eod'], row['f1']),
                textcoords='offset points', xytext=(5, 5), fontsize=8)
ax.set_xlabel('Equal Opportunity Difference (lower = fairer)')
ax.set_ylabel('Overall F1 Score')
ax.set_title('Accuracy-Fairness Pareto Frontier\n(ThresholdOptimizer with equalized_odds)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('part4_pareto_frontier.png', dpi=150)
plt.show()

In [ ]:
# Oversample high-black cohort 3x (so each row appears 4 times total)
high_black_train_rows = df_train[high_black_train].copy()
df_train_oversample = pd.concat(
    [df_train] + [high_black_train_rows] * 3,
    ignore_index=True
)
df_train_oversample = df_train_oversample.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Original training set: {len(df_train)} rows")
print(f"Oversampled training set: {len(df_train_oversample)} rows")
print(f"High-black rows (original): {high_black_train.sum()}")
print(f"High-black rows (oversampled): {(df_train_oversample['black'] >= 0.5).sum()}")
print(f"Toxic rate (oversampled): {df_train_oversample['label'].mean():.4f}")

In [ ]:
tokenizer_os = AutoTokenizer.from_pretrained(BASE_MODEL)

os_train_ds = ToxicityDataset(
    df_train_oversample['comment_text'].tolist(),
    df_train_oversample['label'].tolist(),
    tokenizer_os
)
os_eval_ds = ToxicityDataset(
    df_eval['comment_text'].tolist(),
    df_eval['label'].tolist(),
    tokenizer_os
)

model_os = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2)

os_args = TrainingArguments(
    output_dir='./results_oversample',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=200,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    report_to='none',
    seed=SEED,
)

os_trainer = Trainer(
    model=model_os,
    args=os_args,
    train_dataset=os_train_ds,
    eval_dataset=os_eval_ds,
    compute_metrics=compute_metrics,
)

print("Training oversampled model...")
os_trainer.train()
print("Done.")

os_trainer.save_model('./model_checkpoint_oversample')
prob_os = get_probs_from_trainer(os_trainer, os_eval_ds)
np.save('eval_probs_oversample.npy', prob_os)

os_row = compute_fairness_row(prob_os, true_labels, high_black_eval.values, ref_eval.values, 'Oversampling (data aug)')
print("\nOversampling row:", os_row)

In [ ]:
summary_table = pd.DataFrame([
    baseline_row,
    rw_row,
    to_row,
    os_row
])

print("\n" + "=" * 90)
print("MITIGATION COMPARISON TABLE")
print("=" * 90)
print(summary_table.to_string(index=False))

# Save best mitigated model (threshold optimizer is post-hoc and doesn't require retraining)
# Best mitigated: determined by best trade-off in table
# Save oversampled model as best checkpoint to disk
print("\nBest mitigated model checkpoint: ./model_checkpoint_oversample")

In [ ]:
# Visual summary
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

techniques = ['Baseline', 'Reweighing', 'Threshold\nOptimizer', 'Oversampling']
f1_vals = [float(r['Overall F1']) for r in [baseline_row, rw_row, to_row, os_row]]
fpr_black_vals = [float(r['High-Black FPR']) for r in [baseline_row, rw_row, to_row, os_row]]
fpr_ref_vals = [float(r['Reference FPR']) for r in [baseline_row, rw_row, to_row, os_row]]

x = np.arange(len(techniques))
w = 0.3

axes[0].bar(x, f1_vals, color='steelblue', alpha=0.8)
axes[0].set_xticks(x); axes[0].set_xticklabels(techniques)
axes[0].set_ylim(0, 1); axes[0].set_ylabel('F1 Score')
axes[0].set_title('Overall F1 by Technique')
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(x - w/2, fpr_black_vals, w, label='High-Black FPR', color='#e74c3c', alpha=0.8)
axes[1].bar(x + w/2, fpr_ref_vals, w, label='Reference FPR', color='#3498db', alpha=0.8)
axes[1].set_xticks(x); axes[1].set_xticklabels(techniques)
axes[1].set_ylim(0, max(fpr_black_vals + fpr_ref_vals) * 1.3)
axes[1].set_ylabel('False Positive Rate')
axes[1].set_title('FPR by Cohort Across Techniques')
axes[1].legend(); axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('part4_mitigation_summary.png', dpi=150)
plt.show()

## 4.5 Can You Simultaneously Achieve Demographic Parity AND Equalized Odds?

**Short answer: No — not when base rates differ between cohorts.**

**Mathematical proof:**

Let:
- $p_B$ = base rate (prevalence of toxic comments) in high-black cohort
- $p_R$ = base rate in reference cohort  
- $\text{PPR}_B$, $\text{PPR}_R$ = positive prediction rates (demographic parity condition)
- $\text{TPR}_B$, $\text{TPR}_R$ = true positive rates (equalized odds condition on sensitive class)
- $\text{FPR}_B$, $\text{FPR}_R$ = false positive rates (equalized odds condition on non-sensitive class)

**Demographic parity** requires: $\text{PPR}_B = \text{PPR}_R$

Using the law of total probability:
$$\text{PPR} = \text{TPR} \cdot p + \text{FPR} \cdot (1-p)$$

**Equalized odds** requires both $\text{TPR}_B = \text{TPR}_R$ AND $\text{FPR}_B = \text{FPR}_R$.

If equalized odds holds, then:
$$\text{PPR}_B - \text{PPR}_R = \text{TPR}(p_B - p_R) + \text{FPR}[(1-p_B) - (1-p_R)]$$
$$= \text{TPR}(p_B - p_R) - \text{FPR}(p_B - p_R) = (\text{TPR} - \text{FPR})(p_B - p_R)$$

For $\text{PPR}_B = \text{PPR}_R$ (demographic parity), we need:
$$(\text{TPR} - \text{FPR})(p_B - p_R) = 0$$

This is only zero if: (a) $p_B = p_R$ (equal base rates), OR (b) $\text{TPR} = \text{FPR}$ (a random classifier).

**Observed base rates:**
- Check actual values from the data cells above

Since the base rates differ (toxic prevalence is different in the high-black vs reference cohort), it is **mathematically impossible** to satisfy both demographic parity and equalized odds with any non-trivial classifier. This is the Chouldechova impossibility theorem (2017).

**Implication for platform policy**: The choice between demographic parity and equalized odds is a policy decision, not a technical one. Equalized odds (equal error rates) is generally preferred in content moderation because it directly measures whether the model makes equal mistakes across groups, which is what the civil rights complaint was about.